In [1]:
#cosine part

In [3]:
import numpy as np
from scipy.special import jv
from numpy.polynomial.chebyshev import Chebyshev
from scipy.optimize import minimize
from functools import reduce

# ----- Symbolic Polynomial Algebra -----

def symbolic_poly_multiply(p1, p2):
    result_len = len(p1) + len(p2) - 1
    result = [0j] * result_len
    for i in range(len(p1)):
        for j in range(len(p2)):
            result[i + j] += p1[i] * p2[j]
    return result

def symbolic_poly_multiply_offdiagonal(p1, p2):
    result_len = len(p1) + len(p2) + 1
    result = [0j] * result_len
    for i in range(len(p1)):
        for j in range(len(p2)):
            term = p1[i] * p2[j]
            result[i + j + 1] += term
            result[i + j + 2] -= term
    return result

def add_symbolic_polynomials(p1, p2):
    max_len = max(len(p1), len(p2))
    p1 += [0j] * (max_len - len(p1))
    p2 += [0j] * (max_len - len(p2))
    return [a + b for a, b in zip(p1, p2)]

# ----- Symbolic Matrix Operations -----

def multiply_symbolic_matrices(A, B):
    R = np.empty((2, 2), dtype=object)
    for i in range(2):
        for j in range(2):
            result = [0j]
            for k in range(2):
                if i != k and j != k:
                    prod = symbolic_poly_multiply_offdiagonal(B[i, k], A[k, j])
                else:
                    prod = symbolic_poly_multiply(B[i, k], A[k, j])
                result = add_symbolic_polynomials(result, prod)
            R[i, j] = result
    return R

# ----- Generate 2x2 Symbolic Matrix from f -----

def create_numeric_matrix(f, idx):
    A = np.empty((2, 2), dtype=object)

    f1, f2 = f[2 * idx], f[2 * idx + 1]
    barf1 = np.conj(f1)

    a00 = barf1 * f2
    a01 = f1 * f2 - barf1 * f2
    a10 = a01
    a20 = -np.conj(a01)
    a30 = np.conj(a00)
    a31 = np.conj(a01)

    A[0, 0] = [a00, a01]
    A[0, 1] = [a10]
    A[1, 0] = [a20]
    A[1, 1] = [a30, a31]
    return A

# ----- Target Coefficients from Chebyshev Series -----

def target_coeffs(t, max_k, max_degree):
    coeff_total = np.zeros(2 * max_degree)
    T = np.zeros((2 * max_degree, 2 * max_degree))
    T[0][0] = 1
    T[1][1] = 1
    for i in range(2, len(T)):
        T[i,:] = 2 * np.concatenate((np.zeros(1),T[i-1,0:len(T[0])-1]))
        T[i,:] -= T[i-2,:]
    coeff_total = jv(0, t) * T[0, :]
    for k in range(2, len(T), 2):
    #     n = 2 * k
    #     J_val = jv(n, t)
    #     factor = (-1) ** k * 2 * J_val

        # # Get Chebyshev T_{2k}(x), convert to power basis
        # Tn = Chebyshev.basis(n)
        # coeffs = Tn.convert(kind=np.polynomial.Polynomial).coef
        # padded = np.pad(coeffs, (0, max_degree + 1 - len(coeffs)))

        # coeff_total += factor * padded
        coeff_total += 2 * (-1)**(int(k/2)) * jv(k, t) * T[k, :] 
    return coeff_total

# ----- Cost Function -----

def compute_cost(phases, t):
    # Create f_i = exp(2j * phi_i)
    f = [np.exp(1j * phi) for phi in phases]

    # Create symbolic matrices
    matrices = [create_numeric_matrix(f, i) for i in range(len(f) // 2)]

    # Multiply all matrices: R = A0 * A1 * A2 * ...
    R = reduce(multiply_symbolic_matrices, matrices)

    # Extract polynomial from R[0,0]
    R00 = R[0, 0]

    # Get target coefficients
    target = target_coeffs(t, len(R00)-1, (len(R00) - 1) * 2)
    target = target[0::2]
    # print(R00)
    # print(target)

    # Compute cost: sum of squared real-part differences for even powers only
    cost = 0.0
    for i in range(0, min(len(R00), len(target))):
        cost += (R00[i].real - target[i]) ** 2
    return cost

def compute_cost_Test(phases, t):
    # Create f_i = exp(2j * phi_i)
    f = [np.exp(1j * phi) for phi in phases]

    # Create symbolic matrices
    matrices = [create_numeric_matrix(f, i) for i in range(len(f) // 2)]

    # Multiply all matrices: R = A0 * A1 * A2 * ...
    R = reduce(multiply_symbolic_matrices, matrices)

    # Extract polynomial from R[0,0]
    R00 = R[0, 0]
    
    # Get target coefficients
    target = target_coeffs(t, len(R00)-1, (len(R00) - 1) * 2)
    target = target[0::2]
    print(R00)
    print(target)

    # Compute cost: sum of squared real-part differences for even powers only
    cost = 0.0
    for i in range(0, min(len(R00), len(target))):
        cost += (R00[i].real - target[i]) ** 2
    return cost


# ----- Run Optimization -----

if __name__ == "__main__":
    # Parameters
    t = 1.2
    num_phases = 6 # 3 matrices => 6 phases

    # Initial guess for phases
    initial_phases = np.random.rand(num_phases)
    # Minimize cost
    result = minimize(compute_cost, initial_phases, args=(t), method='BFGS')

    # Output results
    print("Optimized phases:")
    print(result.x)
    print("\nMinimum cost:")
    print(result.fun)

    compute_cost_Test(result.x, t)

Optimized phases:
[ 7.21727145e-01 -5.33616638e-01  6.36357140e-01  1.13485647e+00
 -7.60240697e-01  1.03863433e-03]

Minimum cost:
9.895948252044428e-11
[(0.9999901659297081+0.004434866838448792j), (-0.7199992540388499-1.1416750774815256j), (0.08639920671283086-1.9225805077596487j), (-0.0041461300380691846+3.991904026239568j)]
[ 1.00000000e+00 -7.19999999e-01  8.63999925e-02 -4.14716815e-03
  1.06580707e-04 -1.65126474e-06]
